# R515B Soğutucusu için STGP ve EF Tabanlı Özellik Zenginleştirme ve Regresyon Performans Analizi

Bu çalışma, R515B termodinamik soğutucu akışkanının doymuş ve kızgın fazlarındaki veri setlerinde, evrimsel özellik üretiminin (STGP ve EF) regresyon performansına ve termodinamik uyuma (monotonluk) etkisini inceler.

**Senaryolar**

| Senaryo | Girdi uzayı |
|---|---|
| `Base` | Ham girdiler |
| `STGP` | Ham girdiler + STGP ile üretilen öznitelikler |
| `EF` | Ham girdiler + EF (Evolutionary Forest) ile üretilen öznitelikler |
| `HYBRID` | Ham girdiler + STGP + EF öznitelikleri |
| `Domain_Knowledge` *(yalnızca doymuş faz, ayrı karşılaştırma)* | Ham girdi + 1/T + ln(T) |

`Domain_Knowledge` senaryosu, kazancın evrimsel özniteliklerden mi yoksa yalnızca termodinamik dönüşümlerden mi geldiğini ayırt etmek için eklenmiştir. STGP, EF ve HYBRID öznitelikleri yalnızca ham girdilerden üretilir.

**Yöntem**
- **9 farklı regresyon algoritması** (AdaBoost, CatBoost, DART, ET, GBDT, KNN, LightGBM, RF, XGBoost) **5-Fold Çapraz Doğrulama** ile değerlendirilir.
- **Veri sızıntısı önlemi:** Ölçekleyici, STGP, EF ve regresörler her katta **yalnızca o katın eğitim verisiyle** fit edilir. Test verisi ve sentetik ızgara yalnızca `transform`/`predict` görür. Tüm veriyle fit edilen herhangi bir öznitelik veya model sonuçlarda kullanılmaz.
- Üretilen öznitelikler "Evrensel Permutation Importance" ile (test katı üzerinde) değerlendirilir.
- Modellerin termodinamik tutarlılığı, monotonluk ihlal oranı ile sentetik ızgara üzerinde ölçülür; oranlar katlar üzerinden ortalanır. Yöntem fizik bilgili (PIML) olarak nitelendirilmez; ihlal oranları bir uyum ölçütüdür ve hibrit senaryoda artabilir.

*Tüm analizler tek bir K-Fold döngüsünde gerçekleştirilir ve sonuçlar `Results` klasörüne çok sekmeli Excel formatında kaydedilir.*

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from Functions2 import (
    SATURATED_INPUTS, SATURATED_OUTPUTS,
    SUPERHEATED_INPUTS, SUPERHEATED_OUTPUTS,
    run_unified_analysis, summarize_scenarios
)

In [ ]:
# Uyarı: Kendi dizininize ve dosya adınıza göre veri okuma kodunu güncelleyebilirsiniz.
# Örnek okuma kodları:
df_doymus = pd.read_csv("R515B_Ozellikler\R515B_Doymus_Ozellikleri.csv") 
df_kizgin = pd.read_csv("R515B_Ozellikler\R515B_Kizgin_Buhar.csv") 

print("Doymuş Faz Veri Boyutu:", df_doymus.shape)
print("Kızgın Faz Veri Boyutu:", df_kizgin.shape)

In [ ]:
# DOYMUŞ FAZ ANALİZİ (Base, STGP, EF, HYBRID + Domain_Knowledge)
sat_metrics, sat_importances, sat_uyum = run_unified_analysis(
    df=df_doymus,
    input_cols=SATURATED_INPUTS,
    outputs_list=SATURATED_OUTPUTS,
    dataset_name="Doymus_Faz"
)

print("\n--- Doymuş Faz: Senaryo Bazında Ortalama Skorlar ---")
display(summarize_scenarios(sat_metrics, sat_uyum))

print("\n--- Doymuş Faz: Monotonluk İhlali En Düşük Olan Modeller (İlk 15) ---")
display(sat_uyum.sort_values(by=['Ihlal_Yuzdesi_(%)', 'Target']).head(15))

In [ ]:
# KIZGIN FAZ ANALİZİ (Base, STGP, EF, HYBRID)
sup_metrics, sup_importances, sup_uyum = run_unified_analysis(
    df=df_kizgin,
    input_cols=SUPERHEATED_INPUTS,
    outputs_list=SUPERHEATED_OUTPUTS,
    dataset_name="Kizgin_Faz"
)

print("\n--- Kızgın Faz: Senaryo Bazında Ortalama Skorlar ---")
display(summarize_scenarios(sup_metrics, sup_uyum))

print("\n--- Kızgın Faz: Monotonluk İhlali En Düşük Olan Modeller (İlk 15) ---")
display(sup_uyum.sort_values(by=['Ihlal_Yuzdesi_(%)', 'Target']).head(15))